In [1]:
import ast
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
# Retrieval-Ergebnisse laden
retrieval_df = pd.read_csv("data/retrieval_bm25_topic.csv")

retrieval_df["texts"] = retrieval_df["texts"].apply(ast.literal_eval)
retrieval_df["docnos"] = retrieval_df["docnos"].apply(ast.literal_eval)

print("Retrieval-Zeilen:", len(retrieval_df))

Retrieval-Zeilen: 752


In [4]:
# TF-IDF-Funktionen
# Die Extraktion erfolgt über alle ausgewählten Dokumente gemeinsam;
# Termgewichte werden gemittelt, sodass Begriffe hervorgehoben werden,
# die insgesamt für die gesamte Dokumentbasis relevant sind.
def tfidf_top_terms(doc_texts, top_n=5):
    doc_texts = [str(t) for t in doc_texts if t and str(t).strip()]
    if not doc_texts:
        return pd.DataFrame(columns=["term", "tfidf_score"])
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_df=0.9,
        min_df=1,
        max_features=5000
    )
    X = vectorizer.fit_transform(doc_texts)
    scores = np.asarray(X.mean(axis=0)).ravel()
    terms = vectorizer.get_feature_names_out()
    df = pd.DataFrame({"term": terms, "tfidf_score": scores})
    return df.sort_values("tfidf_score", ascending=False).head(top_n)

def tfidf_to_string(tfidf_df):
    return ", ".join(
        f"{r.term} ({r.tfidf_score:.3f})"
        for r in tfidf_df.itertuples(index=False)
    )

def build_evidence(query_text: str, tfidf_df: pd.DataFrame) -> str:
    terms_with_scores = tfidf_to_string(tfidf_df)
    evidence = (
        "Search query:\n"
        f"{query_text}\n\n"
        "Important TF-IDF terms extracted from retrieved documents:\n"
        f"{terms_with_scores}"
    )
    return evidence

In [8]:
# Evidence-Vektoren aus TF-IDF-Terms aufbauen
# rel + non_rel separat, dann contrastive durch Merge beider Varianten

TFIDF_TOP_N = 5

evidence_rows = []

for _, row in retrieval_df.iterrows():
    tfidf_df = tfidf_top_terms(row["texts"], top_n=TFIDF_TOP_N)
    evidence = build_evidence(row["query_text"], tfidf_df)

    evidence_rows.append({
        "session_id": row["session_id"],
        "doc_variant": row["doc_variant"],
        "query_text": row["query_text"],
        "docnos": row["docnos"],
        "tfidf_terms": tfidf_to_string(tfidf_df),
        "evidence": evidence
    })

base_evidence_df = pd.DataFrame(evidence_rows)

# Contrastive Evidence aus rel + non_rel
rel_df = base_evidence_df[base_evidence_df["doc_variant"] == "rel"]
non_rel_df = base_evidence_df[base_evidence_df["doc_variant"] == "non_rel"]

merged = rel_df.merge(
    non_rel_df,
    on=["session_id"],
    suffixes=("_rel", "_non_rel")
)

contrastive_rows = []

for _, row in merged.iterrows():
    evidence = (
        "Search query:\n"
        f"{row['query_text_rel']}\n\n"
        "Important TF-IDF terms extracted from pseudo-relevant documents:\n"
        f"{row['tfidf_terms_rel']}\n\n"
        "Important TF-IDF terms extracted from non-relevant documents:\n"
        f"{row['tfidf_terms_non_rel']}\n\n"
        "Use the pseudo-relevant terms as positive evidence for the topic. "
        "Use the non-relevant terms as contrastive evidence, meaning they indicate what the topic should avoid."
    )

    contrastive_rows.append({
        "session_id": row["session_id"],
        "doc_variant": "contrastive",
        "query_text": row["query_text_rel"],
        "docnos": {"rel": row["docnos_rel"], "non_rel": row["docnos_non_rel"]},
        "tfidf_terms": {"rel": row["tfidf_terms_rel"], "non_rel": row["tfidf_terms_non_rel"]},
        "evidence": evidence
    })

contrastive_df = pd.DataFrame(contrastive_rows)

evidence_df = pd.concat([base_evidence_df, contrastive_df], ignore_index=True)

print("Anzahl Evidence-Zeilen:", len(evidence_df))
display(evidence_df.head())

evidence_df.groupby(["doc_variant"]).size()

Anzahl Evidence-Zeilen: 1128


,session_id,doc_variant,query_text,docnos,tfidf_terms,evidence
0,d19e7b4de358bcfeb89e06095dc26419,rel,peer to peer communication,"[33894675, 301010747, 300892348]","disability (0.130), peer peer (0.123), communi...",Search query:\npeer to peer communication\n\nI...
1,d19e7b4de358bcfeb89e06095dc26419,non_rel,peer to peer communication,"[294182816, 161854289, 310370055, 45476638]","learning (0.142), peer (0.129), students (0.10...",Search query:\npeer to peer communication\n\nI...
2,00dea7cb26b0df14733b1aa2e48d4189,rel,wattpad reading comprehension,"[11012449, 151378454, 122513221]","respondents (0.153), attitude (0.095), class (...",Search query:\nwattpad reading comprehension\n...
3,00dea7cb26b0df14733b1aa2e48d4189,non_rel,wattpad reading comprehension,"[304512725, 478191, 299291709, 293036332]","language (0.146), speed (0.097), reading compr...",Search query:\nwattpad reading comprehension\n...
4,69a3ac0c9cf2fb3bc0b2ceb56db2edfe,rel,synthetic biology nasa,"[78479327, 24774160, 292166320]","team (0.229), igem (0.178), brown (0.153), htt...",Search query:\nsynthetic biology nasa\n\nImpor...


doc_variant
contrastive    376
non_rel        376
rel            376
dtype: int64

In [9]:
evidence_df.to_csv("data/evidence_topic_generation.csv", index=False)

print("Notebook 04 fertig.")
print("Gespeichert: data/evidence_topic_generation.csv")

Notebook 04 fertig.
Gespeichert: data/evidence_topic_generation.csv
